# `verl-tool` — Minimal functional tool-server test

This notebook runs **one thing**: the minimal end-to-end functional test in:

- `scripts/test-simple/run_tool_server_and_test.py`

## What is the “agent” here?
In this minimal demo, the “agent” is simply **this notebook/script acting as a client**: it sends an *action* to the tool server and reads back the *observation*.

In full agentic RL training, the agent would be an **LLM policy** that generates those action strings.

## Action space (in this demo)
- The **action space is text**: a single string (the raw model output).
- For the `python_code` tool, a valid action contains Python code in either:
  - `<python>...</python>` tags, or
  - a fenced block like ```python ... ```.

## Observation / done / valid
The tool server responds (per `trajectory_id`) with:
- `observations`: what the tool produced (e.g. stdout/stderr formatted)
- `valids`: whether the action matched a valid tool format
- `dones`: whether the trajectory is finished

## Reward
This minimal test does **not** compute reward. It validates only the tool-environment interface.

No GPUs, no real LLM, no PPO training.


In [5]:
from __future__ import annotations

import os
import sys
import subprocess
from pathlib import Path

HERE = Path.cwd().resolve()
# Expect: <repo>/verl-tool/scripts/test-simple
VERL_TOOL_ROOT = HERE
for _ in range(4):
    if (VERL_TOOL_ROOT / "pyproject.toml").exists() and (VERL_TOOL_ROOT / "verl_tool").exists():
        break
    VERL_TOOL_ROOT = VERL_TOOL_ROOT.parent

print("python=", sys.executable)
print("cwd=", HERE)
print("VERL_TOOL_ROOT=", VERL_TOOL_ROOT)
assert (VERL_TOOL_ROOT / "verl_tool").exists(), "Could not locate verl-tool package root"


python= /Users/lzanda/Desktop/VRTOOL-Framework/verl-tool/.venv/bin/python3
cwd= /Users/lzanda/Desktop/VRTOOL-Framework/verl-tool/scripts/test-simple
VERL_TOOL_ROOT= /Users/lzanda/Desktop/VRTOOL-Framework/verl-tool


In [6]:
# If imports fail, run the install cell below.
try:
    import verl_tool  # noqa: F401
    print("Import OK: verl_tool")
except Exception as e:
    print("Import FAILED: verl_tool")
    print("Error:", repr(e))
    raise


Import OK: verl_tool


In [7]:
# Minimal install into THIS kernel environment (safe to re-run).
install_steps = [
    [sys.executable, "-m", "pip", "install", "-U", "pip"],
    [sys.executable, "-m", "pip", "install", "-e", str(VERL_TOOL_ROOT)],
]

for cmd in install_steps:
    print("$", " ".join(cmd))
    r = subprocess.run(cmd, cwd=str(VERL_TOOL_ROOT), text=True)
    if r.returncode != 0:
        raise SystemExit("Install step failed. See output above.")

print("\nInstall OK")


$ /Users/lzanda/Desktop/VRTOOL-Framework/verl-tool/.venv/bin/python3 -m pip install -U pip
$ /Users/lzanda/Desktop/VRTOOL-Framework/verl-tool/.venv/bin/python3 -m pip install -e /Users/lzanda/Desktop/VRTOOL-Framework/verl-tool
Obtaining file:///Users/lzanda/Desktop/VRTOOL-Framework/verl-tool
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for verl-tool (pyproject.toml): started
  Building editable for verl-tool (pyproject.toml): finished with status 'done'
  Created wheel for verl-tool: filename=verl_tool-0.1.0-0.editable-py3-

## Run the functional test

What to look for:
- Success ends with: **`[PASS] Minimal functional test succeeded.`**
- Failure shows a traceback and a **`--- server log tail ---`** block.

Note: on macOS you may see a shutdown warning like `resource_tracker ... leaked semaphore objects`. If you see `[PASS]`, you can treat that warning as non-blocking.

In [8]:
test_script = VERL_TOOL_ROOT / "scripts" / "test-simple" / "run_tool_server_and_test.py"
assert test_script.exists(), f"Missing: {test_script}"

cmd = [sys.executable, str(test_script), "--port", "0"]
print("$", " ".join(cmd))

res = subprocess.run(cmd, cwd=str(VERL_TOOL_ROOT), text=True, capture_output=True)
print("exit_code=", res.returncode)
print("--- stdout ---")
print(res.stdout)
print("--- stderr ---")
print(res.stderr)

if res.returncode != 0:
    raise SystemExit("Functional test failed. See output above.")


$ /Users/lzanda/Desktop/VRTOOL-Framework/verl-tool/.venv/bin/python3 /Users/lzanda/Desktop/VRTOOL-Framework/verl-tool/scripts/test-simple/run_tool_server_and_test.py --port 0
exit_code= 0
--- stdout ---
[INFO] Starting tool server:
       /Users/lzanda/Desktop/VRTOOL-Framework/verl-tool/.venv/bin/python3 -m verl_tool.servers.tool_server --host 127.0.0.1 --port 60020 --tool_type python_code --workers_per_tool 2 --http h11
[OK] Server is responding at http://127.0.0.1:60020/get_observation
[INFO] Test 1: python_code tool executes code
[OK] Tool executed and returned expected output.
[INFO] Test 2: finish trajectory
[OK] Finish path returned successfully.

[PASS] Minimal functional test succeeded.
       What you validated: tool server + python tool + finish over HTTP.
[INFO] Stopping server...

--- server log tail (last 4000 chars) ---
2026-03-25 05:39:03,018 - __main__ - INFO - Initializing tools: ('python_code',)
2026-03-25 05:39:03,025 - __main__ - INFO - ✓ Initialized tool: python_co